# Telco Customer Churn — Modeling

Trains 5 classifiers (Decision Tree, Random Forest, SVC, KNN, Logistic Regression) on the preprocessed data, tuned with `GridSearchCV` scored on Recall for the positive class (`Churn == 1`), since missing an actual churner is more costly than a false alarm. Saved models and the test split feed `04_Evaluation.ipynb`.

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.metrics import classification_report , make_scorer , recall_score ,accuracy_score, precision_score, f1_score
 

In [2]:
df = pd.read_csv("../data/Telco-Customer-preprocessing-Churn.csv")
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,1,0,0,1,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,1,0,0,0,1,0,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,1,...,1,0,0,1,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,1,0,0,0,1,0,1,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,1,...,1,0,0,1,0,0,0,0,1,0


In [3]:
y = df["Churn"].values
y

array([0, 0, 1, ..., 0, 1, 0], dtype=int64)

In [4]:
x = df.drop("Churn" ,  axis=1 ).values
x

array([[0., 0., 1., ..., 0., 1., 0.],
       [1., 0., 0., ..., 0., 0., 1.],
       [1., 0., 0., ..., 0., 0., 1.],
       ...,
       [0., 0., 1., ..., 0., 1., 0.],
       [1., 1., 1., ..., 0., 0., 1.],
       [1., 0., 0., ..., 0., 0., 0.]])

In [5]:
x_train , x_test , y_train , y_test = train_test_split(x ,y, test_size=0.2 , random_state=4 , stratify=y)
print ("x_data: " , x_train.shape , x_test.shape)
print ("y_data: ",y_train.shape , y_test.shape)

x_data:  (5634, 40) (1409, 40)
y_data:  (5634,) (1409,)


In [6]:
recall_1 = make_scorer(recall_score , pos_label=1)

### Scaling


Features were standardized using `StandardScaler` for models that are sensitive to feature scale.

**SVC, KNN, and Logistic Regression** use the scaled data, while **Decision Tree and Random Forest** are trained on the original unscaled data because tree-based models do not require feature scaling.

To prevent the raw data from being overwritten, separate datasets were created:

* `x_train` and `x_test`: original data for tree-based models
* `x_train_scaled` and `x_test_scaled`: standardized data for SVC, KNN, and Logistic Regression

Both raw and scaled test sets are saved separately so that each model is evaluated using data with the same scale as the data it was trained on.


In [7]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

#  Decision Tree Classifier

In [8]:
from sklearn.tree import DecisionTreeClassifier

tree_model =  DecisionTreeClassifier(class_weight="balanced" ,random_state=4)

grid_param_tree = {
    "criterion" : ['gini', 'entropy', 'log_loss'],
    "max_depth" : [5 ] ,
    "min_samples_split" : [260 ,270 , 280] ,
    "min_samples_leaf" :[65 , 70, 75, 80]
}

grid_tree = GridSearchCV(tree_model , grid_param_tree , cv=3 , scoring=recall_1 , n_jobs=-1)
grid_tree.fit(x_train , y_train)
best_tree_model = grid_tree.best_estimator_


yhat_tree = best_tree_model.predict(x_test)
print(grid_tree.best_params_)
print(classification_report(y_test , yhat_tree))

{'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 65, 'min_samples_split': 260}
              precision    recall  f1-score   support

           0       0.90      0.71      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.73      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.79      0.73      0.75      1409



# Random Forest


In [9]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(class_weight="balanced" , random_state=4)

grid_param_rf = {
    "n_estimators" : [50 , 100 , 150 ],
    "criterion" : ['gini', 'entropy', 'log_loss'],
    "max_depth" : [3 ,5 , 7] ,
    "min_samples_split" : [295, 300 , 305 ] ,
    "min_samples_leaf" :[75, 80 ,85] 
}

grid_rf = GridSearchCV(rf_model , grid_param_rf , cv= 3 , scoring=recall_1 , n_jobs=-1)
grid_rf.fit(x_train , y_train )

best_rf_model = grid_rf.best_estimator_
print(grid_rf.best_params_)
yhat_rf = best_rf_model.predict(x_test)
print( classification_report(y_test , yhat_rf))

{'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 75, 'min_samples_split': 295, 'n_estimators': 50}
              precision    recall  f1-score   support

           0       0.91      0.68      0.78      1035
           1       0.48      0.82      0.61       374

    accuracy                           0.72      1409
   macro avg       0.70      0.75      0.69      1409
weighted avg       0.80      0.72      0.74      1409



# SVC

In [10]:
from sklearn.svm import SVC
svc_model = SVC(class_weight="balanced" ,random_state=4 , probability=True)

grid_param_svc ={
    "C" : [0.05 ,0.1 ,0.3],
    "kernel" :['linear', 'poly', 'rbf', 'sigmoid'],
    "gamma" : ['scale', 'auto']
    }
grid_svc = GridSearchCV(svc_model , grid_param_svc ,cv=3 , scoring=recall_1 , n_jobs=-1)
grid_svc.fit(x_train_scaled, y_train)
best_svc_model = grid_svc.best_estimator_

yhat = best_svc_model.predict(x_test_scaled)
print(classification_report(y_test ,yhat))
print(grid_svc.best_params_)

              precision    recall  f1-score   support

           0       0.92      0.63      0.75      1035
           1       0.45      0.84      0.59       374

    accuracy                           0.68      1409
   macro avg       0.68      0.74      0.67      1409
weighted avg       0.79      0.68      0.70      1409

{'C': 0.05, 'gamma': 'scale', 'kernel': 'poly'}


# KNN

In [11]:
from sklearn.neighbors import KNeighborsClassifier
KNN_model = KNeighborsClassifier(n_neighbors = 17 ,weights="distance" )

n = list(range(1 ,101))
grid_param_knn = {
    "n_neighbors" : n ,
}
grid_knn = GridSearchCV( KNN_model , grid_param_knn ,cv=3 , scoring=recall_1 , n_jobs=-1)
grid_knn.fit(x_train_scaled, y_train)
best_KNN_model = grid_knn.best_estimator_

yhat_knn = best_KNN_model.predict(x_test_scaled)
print(grid_knn.best_params_)
print(classification_report(y_test , yhat_knn))

{'n_neighbors': 89}
              precision    recall  f1-score   support

           0       0.84      0.86      0.85      1035
           1       0.58      0.56      0.57       374

    accuracy                           0.78      1409
   macro avg       0.71      0.71      0.71      1409
weighted avg       0.77      0.78      0.78      1409



# logestic regresion 

In [12]:
from sklearn.linear_model import LogisticRegression
lg_model = LogisticRegression(random_state=4 ,   )

grid_param_lg = [
    {
        "solver": ["liblinear" , "saga"],
        "penalty": ["l1", "l2"],
        "C": [0.3, 0.5, 0.8],
        "class_weight": [None, "balanced"],
        "max_iter": [100, 200],
    },

    {
        "solver": ["lbfgs", "newton-cg", "newton-cholesky", "sag"],
        "penalty": ["l2"],  
        "C": [0.3, 0.5, 0.8],
        "class_weight": [None, "balanced"],
        "max_iter": [100, 200],
    },
]


grid_lg = GridSearchCV(lg_model , grid_param_lg , cv=3 , scoring=recall_1 ,n_jobs=-1)
grid_lg.fit(x_train_scaled, y_train)

best_lg_model = grid_lg.best_estimator_

yhat_lg = best_lg_model.predict(x_test_scaled)
print(grid_lg.best_params_)
print(classification_report(y_test , yhat_lg))



{'C': 0.5, 'class_weight': 'balanced', 'max_iter': 100, 'penalty': 'l1', 'solver': 'liblinear'}
              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1035
           1       0.53      0.81      0.64       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.76      0.77      1409



### Save models

In [13]:
import joblib

joblib.dump(best_tree_model, "../models/decision_tree.pkl")
joblib.dump(best_rf_model, "../models/random_forest.pkl")
joblib.dump(best_svc_model, "../models/svc.pkl")
joblib.dump(best_KNN_model, "../models/knn.pkl")
joblib.dump(best_lg_model, "../models/logistic_regression.pkl")

['../models/logistic_regression.pkl']

### Save test data

Save **both** the raw and the scaled test features, clearly named, so `04_Evaluation.ipynb` can feed each model the scale it was actually trained on — Decision Tree / Random Forest get `x_test_raw.csv`, SVC / KNN / Logistic Regression get `x_test_scaled.csv`.

In [14]:
pd.DataFrame(x_test).to_csv("../data/x_test_raw.csv", index=False)
pd.DataFrame(x_test_scaled).to_csv("../data/x_test_scaled.csv", index=False)
pd.DataFrame(y_test).to_csv("../data/y_test.csv", index=False)